# Grafana Interactive Documentation

A living canvas for exploring the Grafana codebase — architecture, API surface, frontend features, and dependencies.

**Run cells top-to-bottom**, or jump to any section:
1. [Project Overview](#1-project-overview)
2. [Directory Explorer](#2-directory-explorer)
3. [Backend API Routes](#3-backend-api-routes)
4. [Frontend Feature Map](#4-frontend-feature-map)
5. [Go Package Graph](#5-go-package-graph)
6. [Frontend Dependency Map](#6-frontend-dependency-map)
7. [Search & Discover](#7-search--discover)
8. [Configuration Reference](#8-configuration-reference)

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
import os, re, json, subprocess, textwrap
from pathlib import Path
from collections import defaultdict, Counter
from IPython.display import display, HTML, Markdown

ROOT = Path(os.environ.get("GRAFANA_ROOT", "/Users/amrita/Desktop/grafana"))
assert ROOT.exists(), f"Grafana root not found at {ROOT}"

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd or ROOT)
    return r.stdout.strip()

def _html(s):
    display(HTML(s))

def _md(s):
    display(Markdown(s))

CSS = """
<style>
.doc-card{border:1px solid #3a3f47;border-radius:8px;padding:16px 20px;margin:8px 0;
  background:#1e2228;color:#cdd9e5;font-family:system-ui,sans-serif;}
.doc-card h3{margin:0 0 6px;color:#58a6ff;font-size:15px;}
.doc-card p{margin:0;font-size:13px;line-height:1.5;color:#8b949e;}
.doc-card code{background:#2d333b;padding:2px 6px;border-radius:4px;font-size:12px;color:#e6edf3;}
.doc-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(320px,1fr));gap:10px;}
.badge{display:inline-block;padding:2px 8px;border-radius:12px;font-size:11px;font-weight:600;margin-right:4px;}
.badge-go{background:#00add8;color:#fff;}
.badge-ts{background:#3178c6;color:#fff;}
.badge-react{background:#61dafb;color:#000;}
.badge-config{background:#f0883e;color:#fff;}
.badge-docs{background:#8957e5;color:#fff;}
.tree-table{width:100%;border-collapse:collapse;font-family:'SF Mono',monospace;font-size:13px;}
.tree-table td,.tree-table th{padding:5px 10px;text-align:left;border-bottom:1px solid #30363d;color:#cdd9e5;}
.tree-table th{color:#8b949e;font-weight:600;font-size:12px;text-transform:uppercase;letter-spacing:.5px;}
.tree-table tr:hover{background:#161b22;}
.search-box{width:100%;padding:10px 14px;border:1px solid #3a3f47;border-radius:8px;
  background:#0d1117;color:#e6edf3;font-size:14px;font-family:system-ui;margin-bottom:12px;}
.metric{font-size:28px;font-weight:700;color:#58a6ff;}
.metric-label{font-size:12px;color:#8b949e;text-transform:uppercase;letter-spacing:.5px;}
.metric-card{text-align:center;padding:20px;border:1px solid #30363d;border-radius:8px;background:#161b22;}
</style>
"""
_html(CSS)
print(f"✔ Grafana root: {ROOT}")

---
## 1. Project Overview <a id='1-project-overview'></a>

In [ ]:
pkg_json = json.loads((ROOT / "package.json").read_text())
go_mod_ver = ""
for line in (ROOT / "go.mod").read_text().splitlines():
    if line.strip().startswith("go "):
        go_mod_ver = line.strip().split()[1]
        break

go_files   = int(_run("find pkg apps -name '*.go' | wc -l"))
ts_files   = int(_run("find public/app packages -name '*.ts' -o -name '*.tsx' 2>/dev/null | wc -l"))
test_files = int(_run("find . -name '*_test.go' -o -name '*.test.ts' -o -name '*.test.tsx' 2>/dev/null | wc -l"))

git_branch = _run("git rev-parse --abbrev-ref HEAD")
last_commit = _run('git log -1 --format="%h — %s (%ar)"')

_html(f"""
<div class="doc-grid">
  <div class="metric-card"><div class="metric">{pkg_json['version']}</div><div class="metric-label">Version</div></div>
  <div class="metric-card"><div class="metric">Go {go_mod_ver}</div><div class="metric-label">Backend</div></div>
  <div class="metric-card"><div class="metric">{go_files:,}</div><div class="metric-label">Go Files</div></div>
  <div class="metric-card"><div class="metric">{ts_files:,}</div><div class="metric-label">TS/TSX Files</div></div>
  <div class="metric-card"><div class="metric">{test_files:,}</div><div class="metric-label">Test Files</div></div>
  <div class="metric-card"><div class="metric">{git_branch}</div><div class="metric-label">Branch</div></div>
</div>
<br/>
<div class="doc-card"><h3>Latest Commit</h3><p><code>{last_commit}</code></p></div>
<div class="doc-card"><h3>License</h3><p>{pkg_json.get('license', 'N/A')}</p></div>
""")

---
## 2. Directory Explorer <a id='2-directory-explorer'></a>

Explore the top-level structure and drill into any directory.

In [ ]:
def explore_directory(rel_path=".", depth=1):
    """Show contents of a directory with file counts and descriptions."""
    target = ROOT / rel_path
    if not target.is_dir():
        print(f"Not a directory: {rel_path}")
        return

    known_dirs = {
        "pkg": ("Go backend packages", "badge-go", "Go"),
        "public": ("Frontend app, assets, Sass", "badge-ts", "Frontend"),
        "packages": ("Shared @grafana/* libraries", "badge-ts", "Libraries"),
        "apps": ("CUE-based app-platform modules", "badge-go", "Apps"),
        "conf": ("Default config & provisioning", "badge-config", "Config"),
        "devenv": ("Docker blocks, compose stacks", "badge-config", "DevEnv"),
        "docs": ("Published documentation sources", "badge-docs", "Docs"),
        "scripts": ("Build, CI, codegen helpers", "badge-config", "Scripts"),
        "e2e": ("Legacy Cypress E2E tests", "badge-ts", "Test"),
        "e2e-playwright": ("Playwright E2E tests", "badge-ts", "Test"),
        "kinds": ("Kind schema definitions", "badge-config", "Schema"),
        "kindsv2": ("v2 Kind schema definitions", "badge-config", "Schema"),
        "emails": ("Email templates", "badge-docs", "Templates"),
        "tools": ("Dev tooling", "badge-config", "Tools"),
        "contribute": ("Contributor documentation", "badge-docs", "Docs"),
        "packaging": ("Packaging configs (deb, rpm, etc.)", "badge-config", "Package"),
    }

    entries = sorted(target.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower()))
    rows = []
    for entry in entries:
        if entry.name.startswith(".") or entry.name == "node_modules":
            continue
        name = entry.name
        if entry.is_dir():
            child_count = sum(1 for _ in entry.iterdir() if not _.name.startswith("."))
            desc, badge_cls, badge_txt = known_dirs.get(name, ("", "badge-config", "Dir"))
            badge = f'<span class="badge {badge_cls}">{badge_txt}</span>'
            rows.append(f"<tr><td>📁 <strong>{name}/</strong></td><td>{badge}</td><td>{desc}</td><td>{child_count} items</td></tr>")
        elif entry.suffix in (".go", ".ts", ".tsx", ".js", ".json", ".md", ".yaml", ".yml", ".toml", ".mod"):
            size = entry.stat().st_size
            size_str = f"{size:,} B" if size < 1024 else f"{size/1024:.1f} KB"
            rows.append(f"<tr><td>📄 {name}</td><td></td><td></td><td>{size_str}</td></tr>")

    header = f"<h3 style='color:#58a6ff;font-family:system-ui;'>📂 {rel_path}/</h3>"
    table = f"""
    <table class="tree-table">
      <tr><th>Name</th><th>Type</th><th>Description</th><th>Size</th></tr>
      {''.join(rows)}
    </table>"""
    _html(header + table)

explore_directory(".")

In [ ]:
# ── Drill into any subdirectory ──
# Change the path below and re-run to explore deeper:
explore_directory("pkg")

---
## 3. Backend API Routes <a id='3-backend-api-routes'></a>

Extracts HTTP routes registered in `pkg/api/api.go` and groups them by method and path prefix.

In [ ]:
def extract_api_routes():
    """Parse route registrations from pkg/api/api.go."""
    api_file = ROOT / "pkg" / "api" / "api.go"
    if not api_file.exists():
        print("pkg/api/api.go not found")
        return []

    content = api_file.read_text()
    route_pattern = re.compile(
        r'\b(r|apiRoute|dashboardRoute)\.(Get|Post|Put|Patch|Delete|Any)\(\s*"([^"]+)"',
        re.MULTILINE,
    )
    routes = []
    for m in route_pattern.finditer(content):
        group, method, path = m.group(1), m.group(2).upper(), m.group(3)
        routes.append({"method": method, "path": path, "group": group})
    return routes


def render_routes(routes, filter_prefix=""):
    if filter_prefix:
        routes = [r for r in routes if r["path"].startswith(filter_prefix)]

    method_colors = {
        "GET": "#238636", "POST": "#1f6feb", "PUT": "#d29922",
        "PATCH": "#d29922", "DELETE": "#da3633", "ANY": "#8b949e",
    }

    prefix_groups = defaultdict(list)
    for r in routes:
        parts = r["path"].strip("/").split("/")
        prefix = "/" + "/".join(parts[:2]) if len(parts) > 1 else "/" + parts[0]
        prefix_groups[prefix].append(r)

    _md(f"**{len(routes)} routes found** (filter: `{filter_prefix or '*'}`)")

    rows = []
    for prefix in sorted(prefix_groups):
        for r in sorted(prefix_groups[prefix], key=lambda x: x["path"]):
            color = method_colors.get(r["method"], "#8b949e")
            method_badge = f'<span style="background:{color};color:#fff;padding:1px 6px;border-radius:4px;font-size:11px;font-weight:700;">{r["method"]}</span>'
            rows.append(f"<tr><td>{method_badge}</td><td><code>{r['path']}</code></td><td>{prefix}</td></tr>")

    _html(f"""
    <table class="tree-table">
      <tr><th>Method</th><th>Path</th><th>Group</th></tr>
      {''.join(rows)}
    </table>""")


routes = extract_api_routes()
render_routes(routes)

In [ ]:
# ── Filter routes by prefix ──
# Change the prefix below to filter (e.g. "/api/dashboards", "/api/datasources", "/api/org"):
render_routes(routes, filter_prefix="/api/dashboards")

---
## 4. Frontend Feature Map <a id='4-frontend-feature-map'></a>

Scans `public/app/features/` and generates a card for each feature area.

In [ ]:
def frontend_feature_map():
    features_dir = ROOT / "public" / "app" / "features"
    if not features_dir.is_dir():
        print("public/app/features/ not found")
        return

    features = []
    for d in sorted(features_dir.iterdir()):
        if not d.is_dir() or d.name.startswith("."):
            continue
        tsx_count = len(list(d.rglob("*.tsx")))
        ts_count = len(list(d.rglob("*.ts")))
        test_count = len(list(d.rglob("*.test.*")))
        has_index = (d / "index.ts").exists() or (d / "index.tsx").exists()
        features.append({
            "name": d.name,
            "tsx": tsx_count,
            "ts": ts_count,
            "tests": test_count,
            "has_index": has_index,
        })

    features.sort(key=lambda f: f["tsx"] + f["ts"], reverse=True)

    cards = []
    for f in features:
        total = f["tsx"] + f["ts"]
        bar_width = min(100, int(total / max(1, features[0]["tsx"] + features[0]["ts"]) * 100))
        test_ratio = f"{f['tests']}/{total}" if total else "0/0"
        cards.append(f"""
        <div class="doc-card">
          <h3>{f['name']}</h3>
          <p>
            <code>{f['tsx']} tsx</code> · <code>{f['ts']} ts</code> · <code>{f['tests']} tests</code>
            &nbsp;·&nbsp; test ratio: {test_ratio}
          </p>
          <div style="margin-top:8px;height:4px;background:#21262d;border-radius:2px;">
            <div style="width:{bar_width}%;height:100%;background:#58a6ff;border-radius:2px;"></div>
          </div>
        </div>""")

    _md(f"**{len(features)} feature modules** in `public/app/features/`")
    _html(f'<div class="doc-grid">{chr(10).join(cards)}</div>')

frontend_feature_map()

---
## 5. Go Package Graph <a id='5-go-package-graph'></a>

Maps Go packages under `pkg/` and shows their import relationships as a dependency table.

In [ ]:
def go_package_analysis(base="pkg", max_depth=1):
    """Analyze Go packages: LOC, import counts, internal dependencies."""
    pkg_dir = ROOT / base
    if not pkg_dir.is_dir():
        print(f"{base}/ not found")
        return

    packages = []
    for d in sorted(pkg_dir.iterdir()):
        if not d.is_dir() or d.name.startswith("."):
            continue
        go_files = list(d.rglob("*.go"))
        test_files = [f for f in go_files if f.name.endswith("_test.go")]
        src_files = [f for f in go_files if not f.name.endswith("_test.go")]

        loc = 0
        internal_imports = set()
        for f in src_files[:200]:  # cap for speed
            try:
                text = f.read_text(errors="ignore")
                loc += text.count("\n")
                for m in re.findall(r'"github\.com/grafana/grafana/pkg/(\w+)', text):
                    if m != d.name:
                        internal_imports.add(m)
            except Exception:
                pass

        packages.append({
            "name": d.name,
            "files": len(src_files),
            "tests": len(test_files),
            "loc": loc,
            "imports": sorted(internal_imports),
        })

    packages.sort(key=lambda p: p["loc"], reverse=True)

    rows = []
    for p in packages:
        imp_str = ", ".join(f"<code>{i}</code>" for i in p["imports"][:8])
        if len(p["imports"]) > 8:
            imp_str += f" +{len(p['imports'])-8} more"
        rows.append(f"""
        <tr>
          <td><strong>{p['name']}</strong></td>
          <td>{p['files']}</td>
          <td>{p['tests']}</td>
          <td>{p['loc']:,}</td>
          <td style="font-size:11px;">{imp_str}</td>
        </tr>""")

    _md(f"**{len(packages)} packages** under `{base}/`")
    _html(f"""
    <table class="tree-table">
      <tr><th>Package</th><th>Source Files</th><th>Test Files</th><th>LOC</th><th>Internal Deps</th></tr>
      {''.join(rows)}
    </table>""")

go_package_analysis("pkg")

---
## 6. Frontend Dependency Map <a id='6-frontend-dependency-map'></a>

Analyzes `@grafana/*` packages and their dependency relationships.

In [ ]:
def frontend_dependency_map():
    packages_dir = ROOT / "packages"
    if not packages_dir.is_dir():
        print("packages/ not found")
        return

    pkgs = []
    for d in sorted(packages_dir.iterdir()):
        pkg_json_path = d / "package.json"
        if not pkg_json_path.exists():
            continue
        try:
            pj = json.loads(pkg_json_path.read_text())
        except Exception:
            continue

        name = pj.get("name", d.name)
        version = pj.get("version", "?")
        deps = pj.get("dependencies", {})
        peer_deps = pj.get("peerDependencies", {})

        grafana_deps = [k for k in {**deps, **peer_deps} if k.startswith("@grafana/")]
        ts_files = len(list(d.rglob("*.ts"))) + len(list(d.rglob("*.tsx")))

        pkgs.append({
            "name": name,
            "version": version,
            "grafana_deps": grafana_deps,
            "dep_count": len(deps),
            "files": ts_files,
        })

    pkgs.sort(key=lambda p: p["files"], reverse=True)

    cards = []
    for p in pkgs:
        dep_badges = " ".join(f'<code>{d.replace("@grafana/", "")}</code>' for d in p["grafana_deps"])
        cards.append(f"""
        <div class="doc-card">
          <h3>{p['name']} <span style="font-size:11px;color:#8b949e;">v{p['version']}</span></h3>
          <p><code>{p['files']} files</code> · <code>{p['dep_count']} deps</code></p>
          <p style="margin-top:6px;">Grafana deps: {dep_badges or '<em>none</em>'}</p>
        </div>""")

    _md(f"**{len(pkgs)} packages** under `packages/`")
    _html(f'<div class="doc-grid">{chr(10).join(cards)}</div>')

frontend_dependency_map()

---
## 7. Search & Discover <a id='7-search--discover'></a>

Search for symbols, files, or patterns across the codebase.

In [ ]:
def search_codebase(pattern, file_glob="*.go", max_results=30):
    """Search for a regex pattern across the codebase using ripgrep."""
    cmd = f'rg --no-heading --line-number --color=never -g "{file_glob}" "{pattern}" | head -{max_results}'
    output = _run(cmd)
    if not output:
        _md(f"No results for `{pattern}` in `{file_glob}`")
        return

    lines = output.strip().split("\n")
    rows = []
    for line in lines:
        parts = line.split(":", 2)
        if len(parts) >= 3:
            file_path, line_no, content = parts[0], parts[1], parts[2].strip()
            if len(content) > 120:
                content = content[:120] + "…"
            content = content.replace("<", "&lt;").replace(">", "&gt;")
            rows.append(f"<tr><td><code>{file_path}</code></td><td>{line_no}</td><td style='font-size:12px;'>{content}</td></tr>")

    _md(f"**{len(rows)} results** for `{pattern}` in `{file_glob}`")
    _html(f"""
    <table class="tree-table">
      <tr><th>File</th><th>Line</th><th>Content</th></tr>
      {''.join(rows)}
    </table>""")


# ── Try a search ──
# Change the pattern and file type to search for anything:
search_codebase("func.*Dashboard", file_glob="*.go", max_results=20)

In [ ]:
def search_for_component(name, max_results=20):
    """Search for a React component definition."""
    search_codebase(
        f"(export|const|function).*{name}",
        file_glob="*.tsx",
        max_results=max_results,
    )

# Search for a React component:
search_for_component("DashboardPage")

---
## 8. Configuration Reference <a id='8-configuration-reference'></a>

Extracts feature toggles and configuration sections from the codebase.

In [ ]:
def feature_toggles(max_rows=50):
    """Parse feature toggles from the CSV registry."""
    csv_path = ROOT / "pkg" / "services" / "featuremgmt" / "toggles_gen.csv"
    if not csv_path.exists():
        print("Feature toggles CSV not found")
        return

    lines = csv_path.read_text().strip().split("\n")
    if len(lines) < 2:
        print("CSV is empty")
        return

    header = lines[0].split(",")
    rows = []
    for line in lines[1:max_rows+1]:
        cols = line.split(",")
        if len(cols) >= 2:
            name = cols[0].strip()
            stage = cols[1].strip() if len(cols) > 1 else ""
            owner = cols[2].strip() if len(cols) > 2 else ""
            stage_color = {"GA": "#238636", "preview": "#1f6feb", "experimental": "#d29922"}.get(stage, "#8b949e")
            stage_badge = f'<span style="background:{stage_color};color:#fff;padding:1px 6px;border-radius:4px;font-size:11px;">{stage}</span>'
            rows.append(f"<tr><td><code>{name}</code></td><td>{stage_badge}</td><td style='font-size:12px;color:#8b949e;'>{owner}</td></tr>")

    _md(f"**{len(lines)-1} feature toggles** registered (showing {len(rows)})")
    _html(f"""
    <table class="tree-table">
      <tr><th>Toggle Name</th><th>Stage</th><th>Owner</th></tr>
      {''.join(rows)}
    </table>""")

feature_toggles()

In [ ]:
def config_sections():
    """Extract [section] headers from the default INI config."""
    ini_path = ROOT / "conf" / "defaults.ini"
    if not ini_path.exists():
        print("conf/defaults.ini not found")
        return

    content = ini_path.read_text()
    sections = re.findall(r'^\[([^\]]+)\]', content, re.MULTILINE)

    cards = []
    for s in sections:
        section_match = re.search(
            rf'^\[{re.escape(s)}\]\n((?:.*\n)*?)(?=^\[|\Z)',
            content,
            re.MULTILINE,
        )
        key_count = 0
        if section_match:
            body = section_match.group(1)
            key_count = len([l for l in body.split("\n") if "=" in l and not l.strip().startswith(";")])

        cards.append(f"""
        <div class="doc-card">
          <h3>[{s}]</h3>
          <p><code>{key_count} settings</code></p>
        </div>""")

    _md(f"**{len(sections)} configuration sections** in `conf/defaults.ini`")
    _html(f'<div class="doc-grid">{chr(10).join(cards)}</div>')

config_sections()

---
## Quick Reference

| Task | Cell to Re-run |
|------|----------------|
| Explore a directory | Change path in `explore_directory("pkg/services")` |
| Filter API routes | Change prefix in `render_routes(routes, "/api/org")` |
| Search Go code | `search_codebase("pattern", "*.go")` |
| Search React components | `search_for_component("ComponentName")` |
| Search TS/TSX code | `search_codebase("pattern", "*.tsx")` |
| Feature toggles | `feature_toggles(max_rows=100)` |